# Analise de vendas - Cafeteria

## Objetivo do Projeto

O objetivo desse projeto é iniciar o tratamento e uma analise exploratoria dos dados de vendas da cafeteria, com o objetivo de entender melhor as vendas e realizar a criacao de um dashboard.

Durante o projeto serão realizadas as etapas de:

- Analise exploratoria dos dados;
- Limpeza e tratamento dos dados;
- Tradução dos dados
- Visualizações dos dados;
- Construcao de um dashboard.

---

# 1. Analise exploratoria dos dados (AED);

## Importando as bibliotecas

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Leitura do dataset

In [4]:
df = pd.read_csv('../data/coffeeshop.csv')

pd.options.display.float_format = '{:.2f}'.format

## Entendimento inicial dos dados

In [ ]:
df.head()

In [ ]:
print(f"O dataset possui {df.shape[0]} linhas e {df.shape[1]} colunas.")

In [ ]:
df.info()

In [ ]:
df.describe()

## Descrição das Colunas do Dataset

| Coluna | Descrição |
|---|---|
| `transaction_id` | Identificador único de cada transação realizada. |
| `transaction_date` | Data em que a compra foi efetuada. |
| `transaction_time` | Horário exato da transação. |
| `store_id` | Identificador único da loja onde a venda ocorreu. |
| `store_location` | Localização ou unidade da cafeteria onde a compra foi realizada. |
| `product_id` | Identificador único do produto vendido. |
| `transaction_qty` | Quantidade de itens comprados na transação. |
| `unit_price` | Preço unitário do produto. |
| `Total_Bill` | Valor total pago na compra (`quantidade × preço unitário`). |
| `product_category` | Categoria geral do produto vendido (ex.: Tea, Coffee, Bakery). |
| `product_type` | Tipo específico do produto dentro da categoria. |
| `product_detail` | Descrição detalhada ou sabor do produto. |
| `Size` | Tamanho do produto vendido (Small, Medium, Large, etc.). |
| `Month Name` | Nome do mês em que a transação ocorreu. |
| `Day Name` | Nome do dia da semana da transação. |
| `Hour` | Hora extraída do horário da transação. |
| `Month` | Número do mês da transação. |
| `Day of Week` | Número correspondente ao dia da semana. |

In [ ]:
coluna_categorica = df.select_dtypes(include=['string']).columns
coluna_numerica = df.select_dtypes(include=['number']).columns

coluna_categorica = coluna_categorica.drop(['transaction_date', 'transaction_time'])

In [ ]:
for coluna in coluna_categorica:
    print(f'Os valores unicos da coluna {coluna} são:')
    for index in df[coluna].unique():
        print(f' - {index}')
        


---

# 2. Limpeza e Tratamento dos Dados

Após a análise inicial do dataset, foi possível observar que não existem valores ausentes, registros duplicados e nem valores despadronizados relevantes que necessitem tratamento.

Dessa forma, esta etapa sera ignorada

OBS: deixei passar um café em especifico que encontrei na tradução dos produtos, então resolvir dar mais uma revisada, ja tirei algumas colunas que não achei necessaria pro dashboard e para evitar tradução errado com meu inglês intermediario irei traduzir somente os termos mais facil

In [ ]:
df.columns

In [ ]:
df = df.drop(['transaction_id', 'store_id', 'product_id', 'product_type'], axis=True)

In [ ]:
df['product_detail'] = df['product_detail'].replace('Jamacian Coffee River', 'Jamaican Coffee River')

---

# 3. Tradução dos Dados

In [ ]:
traducao_valores = { 

    'product_category': {
        'Tea': 'Chá',
        'Coffee': 'Café',
        'Bakery': 'Padaria',
        'Drinking Chocolate': 'Chocolate Quente',
        'Flavours': 'Sabores',
        'Loose Tea': 'Chá a Granel',
        'Packaged Chocolate': 'Chocolate Embalado',
        'Branded': 'Produtos da Marca',
        'Coffee beans': 'Grãos de Café'
    },

    'Size': {
        'Large': 'Grande',
        'Regular': 'Regular',
        'Not Defined': 'Não Definido',
        'Small': 'Pequeno'
    },

    'Month Name': {
        'January': 'Janeiro',
        'February': 'Fevereiro',
        'March': 'Março',
        'April': 'Abril',
        'May': 'Maio',
        'June': 'Junho'
    },

    'Day Name': {
        'Monday': 'Segunda-feira',
        'Tuesday': 'Terça-feira',
        'Wednesday': 'Quarta-feira',
        'Thursday': 'Quinta-feira',
        'Friday': 'Sexta-feira',
        'Saturday': 'Sábado',
        'Sunday': 'Domingo'
    }
}

In [ ]:
traducao_colunas = {
    'transaction_date': 'data_da_transacao',
    'transaction_time': 'hora_da_transacao',
    'store_location': 'localizacao_da_loja',
    'transaction_qty': 'quantidade_vendida',
    'unit_price': 'valor_unitario',
    'Total_Bill': 'valor_total',
    'product_category': 'categoria_do_produto',
    'product_detail': 'nome_do_produto',
    'Size': 'tamanho',
    'Month Name': 'nome_do_mes',
    'Day Name': 'nome_do_dia',
    'Hour': 'hora',
    'Month': 'mes',
    'Day of Week': 'dia_da_semana'
}

In [ ]:
# Tradução dos valores
for coluna, valores in traducao_valores.items():
   df[coluna] = df[coluna].replace(valores)

# Tradução das colunas
df.rename(columns=traducao_colunas, inplace=True)

In [ ]:
df.head()

# 4. Visualisação de dados

In [ ]:
# Contagem das categorias
categorias = df['categoria_do_produto'].value_counts(normalize=True) * 100

# Separando categorias menores que 5%
categorias_ajustadas = categorias.copy()

# Soma das categorias menores que 5%
outros = categorias_ajustadas[categorias_ajustadas < 5].sum()

# Mantendo apenas categorias >= 5%
categorias_ajustadas = categorias_ajustadas[categorias_ajustadas >= 5]

# Adicionando "Outros"
categorias_ajustadas['Outros'] = outros

# Criando gráfico
plt.figure(figsize=(10, 8))

plt.pie(
    categorias_ajustadas,
    labels=categorias_ajustadas.index,
    autopct='%1.1f%%',
    startangle=90
)

plt.title('Distribuição das Categorias de Produtos')

plt.axis('equal')

plt.show()

In [ ]:
# Soma total por categoria
total_categoria = df.groupby('categoria_do_produto')['valor_total'].sum().sort_values(ascending=False)

# Tamanho do gráfico
plt.figure(figsize=(12, 6))

# Gráfico horizontal
sns.barplot(
    x=total_categoria.values,
    y=total_categoria.index
)

# Título e labels
plt.title('Faturamento Total por Categoria')
plt.xlabel('Faturamento')
plt.ylabel('Categoria')

# Mostrar gráfico
plt.show()